# 03 — Silver Transformations

Load data from the Raw layer, clean and standardize it, and persist the result to the Silver layer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

spark = SparkSession.builder \
    .appName("lakehouse-silver") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://lakehouse-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [ ]:
raw_path = "s3a://lakehouse/raw/online_retail/"
silver_path = "s3a://lakehouse/silver/online_retail/"

df_raw = spark.read.parquet(raw_path)
df_raw.show(5)

In [ ]:
df_silver = (
    df_raw
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate")))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn("CustomerID", col("CustomerID").cast("string"))
    .filter(col("Quantity").isNotNull())
    .filter(col("UnitPrice").isNotNull())
    .filter(col("InvoiceDate").isNotNull())
    .withColumn("Revenue", col("Quantity") * col("UnitPrice"))
)

df_silver.show(5)

In [ ]:
df_silver.printSchema()
df_silver.count()

In [ ]:
df_silver.write.mode("overwrite").parquet(silver_path)

In [ ]:
df_silver_check = spark.read.parquet(silver_path)
df_silver_check.select("InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice", "Revenue").show(10, truncate=False)

## Result

The Silver layer now contains typed and standardized data ready for analytics.